# Project: BBLF AI Selector v2
# Section: Optimisation & Simulation Tool
# Sub Section: Pre Tournament

Goal: Select the optimal starting 12 players prior to the start of the tournament 

Things to add:
1. Additional constraints to capture fantasy nuances e.g. captain points, vice captain trick
2. Advance simulation based optimisation process 


# 0. Prerequistes

In [21]:
pip install mip==1.16rc0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 
from scipy.stats import norm
import joblib
from concurrent.futures import ProcessPoolExecutor

# Import intermediary functions
from Optimisation_Functions import optimise_fn_efp, optimise_fn_sim_fp

# Set working directory
os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/pre_tourny/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/'
mock_data_dir = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/mock_data/'

In [23]:
# Optimisation Strategy
opt_strat = 'efp' # Options: 'efp' (new player expected fantasy points appraoch), 'sim' (new player fantasy point distribution approach)
current_rnd = 1 # Current Round of the Tournament

# For Simulation Optimisation Strategy
conf_int = 0.6
sim_num = 10

# 1. New EFP Approach

# a. Data Extraction 

In [24]:
# New Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp':
    
    # Data Extraction 
    # Current Round
    current_round = current_rnd

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    efp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_model_score_pre_tourny_short.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 11.68, player_df_raw["exp_pts"] + 4.42)
    player_df_raw["weight"] = 1

    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1

    # Aggregate by player name per round
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Round", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    exp_rnd_points=('exp_points',"sum"),
    games_in_round=('Round',"count"))

    # Availablity Check: Switch to 0 if the team does not play in the round
    # BBL15 Byes: Rnd 4 - Stars, Rnd 5 - Sixers, Rnd 6 - Hurricanes, Rnd 7 - Strikers, Rnd 8 - Sixers, Rnd 9 - Hurricanes
    
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# b. Player points and price prediction for each round

In [25]:
if opt_strat == 'efp':
    # Features required for Pricing Model (Game Round Lags)

    # Loop through all BBL15 players
    pts_per_game = player_df_raw[['Name', 'game_num', 'exp_points']]
    # Create empty dataframe to hold fantasy point outputs from the for loop
    bbl15_game_pts_table_pre = pd.DataFrame()
    bbl15_game_pts_table_pre["Name"] = []
    bbl15_game_pts_table_pre["game_num"] = []
    bbl15_game_pts_table_pre["curr_game_pts"] = []
    bbl15_game_pts_table_pre["prev_game_pts"] = []
    bbl15_game_pts_table_pre["two_prev_game_pts"] = []
    bbl15_game_pts_table_pre["last_2_games_ma_pts"] = []
    bbl15_game_pts_table_pre["last_3_games_ma_pts"] = []
    bbl15_game_pts_table_pre["seas_avg_games_pts"] = []

    # List of all the unique players in the bbl15 season    
    bbl15_play_list = pts_per_game['Name'].unique()

    # For loop for all BBL players in bbl15 season
    with ProcessPoolExecutor() as executor:
        for j in bbl15_play_list:
            full_play_pts_df = pts_per_game[pts_per_game['Name'] == j]
            full_play_pts_df = full_play_pts_df.sort_values(by = ['game_num'])
            full_play_pts_df['games_cnt'] = np.arange(len(full_play_pts_df)) + 1  

            # List of all the rounds played by individual    
            game_num_list = full_play_pts_df['games_cnt'].unique()

            # For loop for all individual rounds by player
            with ProcessPoolExecutor() as executor:
                for k in game_num_list:
                    # Current Game
                    curr_game_play_pts_feat_df = full_play_pts_df[full_play_pts_df['games_cnt'] == k].rename(columns={"exp_points":"curr_game_pts"}).drop(columns=["games_cnt"], axis = 1)

                    # prior game points
                    prior_game_play_pts_df = full_play_pts_df[full_play_pts_df['games_cnt'] == k - 1].rename(columns={"exp_points":"prev_game_pts"})
                    prior_game_play_pts_df = prior_game_play_pts_df.drop(columns=["game_num", "games_cnt"], axis = 1)
                    curr_game_play_pts_feat_df = pd.merge(curr_game_play_pts_feat_df, prior_game_play_pts_df, left_on = ["Name"], right_on = ["Name"], how = "left")

                    # two prior round points
                    two_prior_game_play_pts_df = full_play_pts_df[full_play_pts_df['games_cnt'] == k - 2].rename(columns={"exp_points":"two_prev_game_pts"})
                    two_prior_game_play_pts_df = two_prior_game_play_pts_df.drop(columns=["game_num", "games_cnt"], axis = 1)
                    curr_game_play_pts_feat_df = pd.merge(curr_game_play_pts_feat_df, two_prior_game_play_pts_df, left_on = ["Name"], right_on = ["Name"], how = "left")

                    # Last two rounds moving average
                    if k >= 2:
                        ma_2_games_play_pts_df = full_play_pts_df[(full_play_pts_df['games_cnt'] <= k) & (full_play_pts_df['games_cnt'] >= k - 1)]
                        ma_2_games_play_pts_df = ma_2_games_play_pts_df.drop(columns=["game_num", "games_cnt"], axis = 1)
                        ma_2_games_play_pts_df_agg = ma_2_games_play_pts_df.groupby(["Name"], as_index=False).agg(
                        last_2_games_ma_pts = ('exp_points', "mean"),
                        )

                    else:
                        ma_2_games_play_pts_df_agg = full_play_pts_df[(full_play_pts_df['games_cnt'] == k)]
                        ma_2_games_play_pts_df_agg = ma_2_games_play_pts_df_agg.drop(columns=["game_num", "games_cnt", "exp_points"], axis = 1)
                        ma_2_games_play_pts_df_agg['last_2_games_ma_pts'] = np.nan
                                                                    
                    curr_game_play_pts_feat_df = pd.merge(curr_game_play_pts_feat_df, ma_2_games_play_pts_df_agg, left_on = ["Name"], right_on = ["Name"], how = "left")

                    # Prior 3 Games in the season aggregate attributes
                    if k >= 3:
                        ma_3_games_play_pts_df = full_play_pts_df[(full_play_pts_df['games_cnt'] <= k) & (full_play_pts_df['games_cnt'] >= k - 2)]
                        ma_3_games_play_pts_df = ma_3_games_play_pts_df.drop(columns=["game_num", "games_cnt"], axis = 1)
                        ma_3_games_play_pts_df_agg = ma_3_games_play_pts_df.groupby(["Name"], as_index=False).agg(
                        last_3_games_ma_pts = ('exp_points', "mean"),
                        )

                    else:
                        ma_3_games_play_pts_df_agg = full_play_pts_df[(full_play_pts_df['games_cnt'] == k)]
                        ma_3_games_play_pts_df_agg = ma_3_games_play_pts_df_agg.drop(columns=["game_num", "games_cnt","exp_points"], axis = 1)
                        ma_3_games_play_pts_df_agg['last_3_games_ma_pts'] = np.nan

                    curr_game_play_pts_feat_df = pd.merge(curr_game_play_pts_feat_df, ma_3_games_play_pts_df_agg, left_on = ["Name"], right_on = ["Name"], how = "left")
                    
                    # All prior games in the season aggregate attributes
                    play_seas_df = full_play_pts_df[full_play_pts_df['games_cnt'] < k+1]
                    play_seas_df = play_seas_df.drop(columns=["game_num", "games_cnt"], axis = 1)
                    play_seas_df_agg = play_seas_df.groupby(["Name"], as_index=False).agg(
                    seas_avg_games_pts = ('exp_points', "mean"),
                    )

                    curr_game_play_pts_feat_df = pd.merge(curr_game_play_pts_feat_df, play_seas_df_agg, left_on = ["Name"], right_on = ["Name"], how = "left")

                    # Add all during season attributes to empty table
                    bbl15_game_pts_table_pre = pd.concat([bbl15_game_pts_table_pre, curr_game_play_pts_feat_df])

    # Join round points features to price delta table
    # Add rnd column via player team
    player_team = player_df_raw.groupby(['Name', 'Team'])['exp_points'].count().reset_index().drop(['exp_points'], axis=1)

    bbl15_game_pts_table = pd.merge(bbl15_game_pts_table_pre, player_team, on = 'Name')

    # Get unique team round game counts
    game_rnd_team_df = player_df_raw.groupby(['Team','Round','game_num'])['exp_points'].count().reset_index().drop(['exp_points'], axis=1)
    bbl15_game_pts_table = pd.merge(bbl15_game_pts_table, game_rnd_team_df, on = ['Team', 'game_num']).drop(['Team'], axis=1)

    # For player double gameweek rounds, only return the second game row
    # Identify the max gameweek for each player in each round
    double_GW_index = bbl15_game_pts_table[['Name','Round','game_num']].sort_values(by=['Name', 'Round', 'game_num'], ascending=[True, True, False]).groupby(['Name', 'Round']).nth(0)

    # Select rows which only exist in double GW index
    bbl15_game_pts_table = pd.merge(bbl15_game_pts_table, double_GW_index, on = ['Name', 'Round', 'game_num'], how='inner')

    bbl15_game_pts_table['Round'] = bbl15_game_pts_table['Round'].astype("Int64")
    bbl15_game_pts_table = bbl15_game_pts_table.sort_values(['Name','Round'])

    # Add player price 
    bbl15_game_pts_table = pd.merge(bbl15_game_pts_table, price_df[['Name', 'Price']], on = 'Name', how = 'left').rename(columns={"Price":"price_pre"})

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")


In [26]:
# Create Player EFP & Price for each round
if opt_strat == 'efp':
    # Extract Pricing Models
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    player_df_lags = bbl15_game_pts_table[['Name', 'Round', 'game_num', 'price_pre', 'seas_avg_games_pts', 'last_2_games_ma_pts', 'last_3_games_ma_pts']]

    # For Loop: For each player create rolling price prediction using previous round prediction as new pre price
    # Create empty player df dataframe to hold new player price dataframe
    player_df_new = pd.DataFrame()
    player_df_new["Name"] = []
    player_df_new["Price"] = []
    player_df_new["Team"] = []
    player_df_new["Round"] = []
    player_df_new["Wk_f"] = []
    player_df_new["Bat_f"] = []
    player_df_new["Bowl_f"] = []
    player_df_new["Role"] = []
    player_df_new["weight"] = []
    player_df_new["exp_rnd_points"] = []
    player_df_new["games_in_round"] = []

    for player in player_df['Name'].unique():
        for rnd in player_df_lags[player_df_lags['Name'] == player]['Round'].unique():                  
            # Filter player df lags for current round & player
            curr_rnd_player_df_lags = player_df_lags[(player_df_lags['Round'] == rnd) & (player_df_lags['Name'] == player)]

            # If current round append current row with price prediction
            if rnd == current_round:
                player_df_new_row = player_df[(player_df['Round'] == rnd) & (player_df['Name'] == player)]
                player_df_new = pd.concat([player_df_new, player_df_new_row])
                
            # Check if player plays in the round
            if curr_rnd_player_df_lags.empty:
                print(f"Player {player} does not play in round {rnd}")
                continue

            # Next Available Round
            # unique rounds from player_df_lags
            player_rounds = player_df_lags[player_df_lags['Name'] == player]['Round'].unique()
            # Find next available round
            next_avail_rnd = min([r for r in player_rounds if r > rnd], default=None)
            print(f"Next available round for player {player} after round {rnd} is {next_avail_rnd}")
            if next_avail_rnd is None:
                print(f"No more rounds for player {player} after round {rnd}")
                continue

            # Split into game count dataframes
            curr_rnd_player_df_price_1 = curr_rnd_player_df_lags[curr_rnd_player_df_lags['game_num'] == 1][['Name', 'price_pre','Round','game_num', 'seas_avg_games_pts']]
            curr_rnd_player_df_price_2 = curr_rnd_player_df_lags[curr_rnd_player_df_lags['game_num'] == 2][['Name', 'price_pre','Round','game_num', 'last_2_games_ma_pts']]
            curr_rnd_player_df_price_3 = curr_rnd_player_df_lags[curr_rnd_player_df_lags['game_num'] >= 3][['Name', 'price_pre','Round','game_num', 'last_3_games_ma_pts']]

            # Predict Prices
            if not curr_rnd_player_df_price_1.empty:
                curr_rnd_player_df_price_1['Price_Pred'] = price_model_obj_1.predict(curr_rnd_player_df_price_1[['price_pre','seas_avg_games_pts']])
                curr_rnd_player_df_price_1 = curr_rnd_player_df_price_1[['Name', 'Round', 'game_num', 'Price_Pred']]
                curr_rnd_player_df_price = curr_rnd_player_df_price_1.copy()
            if not curr_rnd_player_df_price_2.empty:
                curr_rnd_player_df_price_2['Price_Pred'] = price_model_obj_2.predict(curr_rnd_player_df_price_2[['price_pre','last_2_games_ma_pts']])
                curr_rnd_player_df_price_2 = curr_rnd_player_df_price_2[['Name', 'Round', 'game_num', 'Price_Pred']]
                curr_rnd_player_df_price = curr_rnd_player_df_price_2.copy()
            if not curr_rnd_player_df_price_3.empty:
                curr_rnd_player_df_price_3['Price_Pred'] = price_model_obj_3.predict(curr_rnd_player_df_price_3[['price_pre','last_3_games_ma_pts']])
                curr_rnd_player_df_price_3 = curr_rnd_player_df_price_3[['Name', 'Round', 'game_num', 'Price_Pred']]
                curr_rnd_player_df_price = curr_rnd_player_df_price_3.copy()
            if curr_rnd_player_df_price_1.empty and curr_rnd_player_df_price_2.empty and curr_rnd_player_df_price_3.empty:
                print("No players in current round")
                continue

            # Update player df lags price pre for next round
            player_df_lags_ind = pd.merge(player_df_lags, curr_rnd_player_df_price, left_on = ["Name", "Round", "game_num"], right_on = ["Name", "Round", "game_num"], how = "inner", suffixes=('', '_new'))
            player_df_lags_ind['price_pre'] = np.where(player_df_lags_ind['Price_Pred'].notnull(), player_df_lags_ind['Price_Pred'], player_df_lags_ind['price_pre'])
            player_df_lags_ind = player_df_lags_ind.drop(columns=['Price_Pred'])
            
            player_df_price = player_df_lags_ind.rename(columns={"price_pre": "Price_Pred"})

            # Increase round by 1 for price prediction to align with next round
            player_df_price['Round'] = next_avail_rnd
            player_df_pred_price = player_df_price[['Name', 'Round', 'Price_Pred']]

            # Merge price predictions back to main player df & append to new df
            player_df_new_row = pd.merge(player_df, player_df_pred_price, left_on = ["Name", "Round"], right_on = ["Name", "Round"], how = "inner")
            
            # Player price is actual price for current round, predicted price for subsequent rounds
            player_df_new_row['Price'] = np.where(player_df_new_row['Round'] == current_rnd, player_df_new_row['Price'], player_df_new_row['Price_Pred'])
            player_df_new_row = player_df_new_row.drop(columns=['Price_Pred'])

            # Append to new dataframe
            player_df_new = pd.concat([player_df_new, player_df_new_row])

    player_df = player_df_new.reset_index(drop=True)
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")  


Next available round for player Aaron Hardie after round 1 is 2
Next available round for player Aaron Hardie after round 2 is 3
Next available round for player Aaron Hardie after round 3 is 4
Next available round for player Aaron Hardie after round 4 is 5
Next available round for player Aaron Hardie after round 5 is 6
Next available round for player Aaron Hardie after round 6 is 7
Next available round for player Aaron Hardie after round 7 is 8
Next available round for player Aaron Hardie after round 8 is 9
Next available round for player Aaron Hardie after round 9 is None
No more rounds for player Aaron Hardie after round 9
Next available round for player Adam Zampa after round 1 is 2
Next available round for player Adam Zampa after round 2 is 3
Next available round for player Adam Zampa after round 3 is 4
Next available round for player Adam Zampa after round 4 is 5
Next available round for player Adam Zampa after round 5 is 6
Next available round for player Adam Zampa after round 6 i

# c. Create separate round df for round based optimisation

In [27]:
# Split player point dataframe by round for optimisation
if opt_strat == 'efp':
    # Create additional rows for players who do not play in every round
    all_rounds = list(range(int(player_df['Round'].min()), int(player_df['Round'].max()) + 1))
    all_players = player_df['Name'].unique()
    full_index = pd.MultiIndex.from_product([all_players, all_rounds], names=['Name', 'Round'])
    player_df = player_df.set_index(['Name', 'Round']).reindex(full_index).reset_index()
    player_df['Price'] = player_df['Price'].ffill()
    player_df['Team'] = player_df['Team'].ffill()
    player_df['Wk_f'] = player_df['Wk_f'].ffill()
    player_df['Bat_f'] = player_df['Bat_f'].ffill()
    player_df['Bowl_f'] = player_df['Bowl_f'].ffill()
    player_df['Role'] = player_df['Role'].ffill()
    player_df['weight'] = player_df['weight'].ffill()
    player_df['exp_rnd_points'] = player_df['exp_rnd_points'].fillna(-100)
    player_df['games_in_round'] = player_df['games_in_round'].fillna(0)
    player_df['Available'] = np.where(player_df['exp_rnd_points'] == -100, 0, player_df['Available'])
    player_df['In_Team'] = player_df['In_Team'].ffill()

    # Split player df by round
    player_df_r1 = player_df[player_df['Round'] == 1].reset_index(drop=True)
    player_df_r2 = player_df[player_df['Round'] == 2].reset_index(drop=True)
    player_df_r3 = player_df[player_df['Round'] == 3].reset_index(drop=True)
    player_df_r4 = player_df[player_df['Round'] == 4].reset_index(drop=True)
    player_df_r5 = player_df[player_df['Round'] == 5].reset_index(drop=True)
    player_df_r6 = player_df[player_df['Round'] == 6].reset_index(drop=True)
    player_df_r7 = player_df[player_df['Round'] == 7].reset_index(drop=True)
    player_df_r8 = player_df[player_df['Round'] == 8].reset_index(drop=True)
    player_df_r9 = player_df[player_df['Round'] == 9].reset_index(drop=True)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")  

# d. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [28]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # a. EFP Optimisation Variables Setup
    # Round 1
    points_r1 = player_df_r1["exp_rnd_points"]
    price_r1 = player_df_r1["Price"]
    weight_r1 = player_df_r1["weight"]
    in_team_r1 = player_df_r1["In_Team"]
    available_r1 = player_df_r1["Available"]
    wk_weight_r1 = player_df_r1["Wk_f"]
    bat_weight_r1 = player_df_r1["Bat_f"]
    bowl_weight_r1 = player_df_r1["Bowl_f"]
    play_cnt_r1, total_player_r1 = 12, range(len(price_r1))
    wk_cnt_r1, total_wk_r1 = 1, range(len(price_r1))
    bat_cnt_r1, total_bat_r1 = 6, range(len(price_r1))
    bowl_cnt_r1, total_bowl_r1 = 5, range(len(price_r1))
    budget_r1, total_budget_r1 = 1783500, range(len(price_r1))

    # Round 2
    points_r2 = player_df_r2["exp_rnd_points"]
    price_r2 = player_df_r2["Price"]
    weight_r2 = player_df_r2["weight"]
    in_team_r2 = player_df_r2["In_Team"]
    available_r2 = player_df_r2["Available"]
    wk_weight_r2 = player_df_r2["Wk_f"]
    bat_weight_r2 = player_df_r2["Bat_f"]
    bowl_weight_r2 = player_df_r2["Bowl_f"]
    play_cnt_r2, total_player_r2 = 12, range(len(price_r2))
    wk_cnt_r2, total_wk_r2 = 1, range(len(price_r2))
    bat_cnt_r2, total_bat_r2 = 6, range(len(price_r2))
    bowl_cnt_r2, total_bowl_r2 = 5, range(len(price_r2))
    budget_r2, total_budget_r2 = 1783500, range(len(price_r2))
    team_play_cnt_r2, total_team_player_r2 = 9, range(len(price_r2)) # At least 9 players from round 1 to be in round 2 team

    # Round 3
    points_r3 = player_df_r3["exp_rnd_points"]
    price_r3 = player_df_r3["Price"]
    weight_r3 = player_df_r3["weight"]
    in_team_r3 = player_df_r3["In_Team"]
    available_r3 = player_df_r3["Available"]
    wk_weight_r3 = player_df_r3["Wk_f"]
    bat_weight_r3 = player_df_r3["Bat_f"]
    bowl_weight_r3 = player_df_r3["Bowl_f"]
    play_cnt_r3, total_player_r3 = 12, range(len(price_r3))
    wk_cnt_r3, total_wk_r3 = 1, range(len(price_r3))
    bat_cnt_r3, total_bat_r3 = 6, range(len(price_r3))
    bowl_cnt_r3, total_bowl_r3 = 5, range(len(price_r3))
    budget_r3, total_budget_r3 = 1783500, range(len(price_r3))
    team_play_cnt_r3, total_team_player_r3 = 9, range(len(price_r3)) # At least 9 players from round 2 to be in round 3 team

    # Round 4
    points_r4 = player_df_r4["exp_rnd_points"]
    price_r4 = player_df_r4["Price"]
    weight_r4 = player_df_r4["weight"]
    in_team_r4 = player_df_r4["In_Team"]
    available_r4 = player_df_r4["Available"]
    wk_weight_r4 = player_df_r4["Wk_f"]
    bat_weight_r4 = player_df_r4["Bat_f"]
    bowl_weight_r4 = player_df_r4["Bowl_f"]
    play_cnt_r4, total_player_r4 = 12, range(len(price_r4))
    wk_cnt_r4, total_wk_r4 = 1, range(len(price_r4))
    bat_cnt_r4, total_bat_r4 = 6, range(len(price_r4))
    bowl_cnt_r4, total_bowl_r4 = 5, range(len(price_r4))
    budget_r4, total_budget_r4 = 1783500, range(len(price_r4))
    team_play_cnt_r4, total_team_player_r4 = 9, range(len(price_r4)) # At least 9 players from round 3 to be in round 4 team

    # Round 5
    points_r5 = player_df_r5["exp_rnd_points"]
    price_r5 = player_df_r5["Price"]
    weight_r5 = player_df_r5["weight"]
    in_team_r5 = player_df_r5["In_Team"]
    available_r5 = player_df_r5["Available"]
    wk_weight_r5 = player_df_r5["Wk_f"]
    bat_weight_r5 = player_df_r5["Bat_f"]
    bowl_weight_r5 = player_df_r5["Bowl_f"]
    play_cnt_r5, total_player_r5 = 12, range(len(price_r5))
    wk_cnt_r5, total_wk_r5 = 1, range(len(price_r5))
    bat_cnt_r5, total_bat_r5 = 6, range(len(price_r5))
    bowl_cnt_r5, total_bowl_r5 = 5, range(len(price_r5))
    budget_r5, total_budget_r5 = 1783500, range(len(price_r5))
    team_play_cnt_r5, total_team_player_r5 = 9, range(len(price_r5)) # At least 9 players from round 4 to be in round 5 team

    # Round 6
    points_r6 = player_df_r6["exp_rnd_points"]
    price_r6 = player_df_r6["Price"]
    weight_r6 = player_df_r6["weight"]
    in_team_r6 = player_df_r6["In_Team"]
    available_r6 = player_df_r6["Available"]
    wk_weight_r6 = player_df_r6["Wk_f"]
    bat_weight_r6 = player_df_r6["Bat_f"]
    bowl_weight_r6 = player_df_r6["Bowl_f"]
    play_cnt_r6, total_player_r6 = 12, range(len(price_r6))
    wk_cnt_r6, total_wk_r6 = 1, range(len(price_r6))
    bat_cnt_r6, total_bat_r6 = 6, range(len(price_r6))
    bowl_cnt_r6, total_bowl_r6 = 5, range(len(price_r6))
    budget_r6, total_budget_r6 = 1783500, range(len(price_r6))
    team_play_cnt_r6, total_team_player_r6 = 9, range(len(price_r6)) # At least 9 players from round 5 to be in round 6 team

    # Round 7
    points_r7 = player_df_r7["exp_rnd_points"]
    price_r7 = player_df_r7["Price"]
    weight_r7 = player_df_r7["weight"]
    in_team_r7 = player_df_r7["In_Team"]
    available_r7 = player_df_r7["Available"]
    wk_weight_r7 = player_df_r7["Wk_f"]
    bat_weight_r7 = player_df_r7["Bat_f"]
    bowl_weight_r7 = player_df_r7["Bowl_f"]
    play_cnt_r7, total_player_r7 = 12, range(len(price_r7))
    wk_cnt_r7, total_wk_r7 = 1, range(len(price_r7))
    bat_cnt_r7, total_bat_r7 = 6, range(len(price_r7))
    bowl_cnt_r7, total_bowl_r7 = 5, range(len(price_r7))
    budget_r7, total_budget_r7 = 1783500, range(len(price_r7))
    team_play_cnt_r7, total_team_player_r7 = 9, range(len(price_r7)) # At least 9 players from round 6 to be in round 7 team

    # Round 8
    points_r8 = player_df_r8["exp_rnd_points"]
    price_r8 = player_df_r8["Price"]
    weight_r8 = player_df_r8["weight"]
    in_team_r8 = player_df_r8["In_Team"]
    available_r8 = player_df_r8["Available"]
    wk_weight_r8 = player_df_r8["Wk_f"]
    bat_weight_r8 = player_df_r8["Bat_f"]
    bowl_weight_r8 = player_df_r8["Bowl_f"]
    play_cnt_r8, total_player_r8 = 12, range(len(price_r8))
    wk_cnt_r8, total_wk_r8 = 1, range(len(price_r8))
    bat_cnt_r8, total_bat_r8 = 6, range(len(price_r8))
    bowl_cnt_r8, total_bowl_r8 = 5, range(len(price_r8))
    budget_r8, total_budget_r8 = 1783500, range(len(price_r8))
    team_play_cnt_r8, total_team_player_r8 = 9, range(len(price_r8)) # At least 9 players from round 7 to be in round 8 team

    # Round 9
    points_r9 = player_df_r9["exp_rnd_points"]
    price_r9 = player_df_r9["Price"]
    weight_r9 = player_df_r9["weight"]
    in_team_r9 = player_df_r9["In_Team"]
    available_r9 = player_df_r9["Available"]
    wk_weight_r9 = player_df_r9["Wk_f"]
    bat_weight_r9 = player_df_r9["Bat_f"]
    bowl_weight_r9 = player_df_r9["Bowl_f"]
    play_cnt_r9, total_player_r9 = 12, range(len(price_r9))
    wk_cnt_r9, total_wk_r9 = 1, range(len(price_r9))
    bat_cnt_r9, total_bat_r9 = 6, range(len(price_r9))
    bowl_cnt_r9, total_bowl_r9 = 5, range(len(price_r9))
    budget_r9, total_budget_r9 = 1783500, range(len(price_r9))
    team_play_cnt_r9, total_team_player_r9 = 9, range(len(price_r9)) # At least 9 players from round 8 to be in round 9 team

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

In [29]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    sel_player_df, sel_player_df_r1, sel_player_df_r2, sel_player_df_r3, sel_player_df_r4, sel_player_df_r5, sel_player_df_r6, sel_player_df_r7,sel_player_df_r8, sel_player_df_r9 = optimise_fn_efp(
        # Round 1
        points_r1, price_r1, weight_r1, in_team_r1, available_r1, wk_weight_r1, bat_weight_r1, bowl_weight_r1,
        play_cnt_r1, total_player_r1, wk_cnt_r1, total_wk_r1, bat_cnt_r1, total_bat_r1, bowl_cnt_r1, total_bowl_r1,
        budget_r1, total_budget_r1, player_df_r1,
        # Round 2
        points_r2, price_r2, weight_r2, in_team_r2, available_r2, wk_weight_r2, bat_weight_r2, bowl_weight_r2,
        play_cnt_r2, total_player_r2, wk_cnt_r2, total_wk_r2, bat_cnt_r2, total_bat_r2, bowl_cnt_r2, total_bowl_r2,
        budget_r2, total_budget_r2, team_play_cnt_r2, total_team_player_r2, player_df_r2,
        # Round 3
        points_r3, price_r3, weight_r3, in_team_r3, available_r3, wk_weight_r3, bat_weight_r3, bowl_weight_r3,
        play_cnt_r3, total_player_r3, wk_cnt_r3, total_wk_r3, bat_cnt_r3, total_bat_r3, bowl_cnt_r3, total_bowl_r3,
        budget_r3, total_budget_r3, team_play_cnt_r3, total_team_player_r3, player_df_r3,
        # Round 4
        points_r4, price_r4, weight_r4, in_team_r4, available_r4, wk_weight_r4, bat_weight_r4, bowl_weight_r4,
        play_cnt_r4, total_player_r4, wk_cnt_r4, total_wk_r4, bat_cnt_r4, total_bat_r4, bowl_cnt_r4, total_bowl_r4,
        budget_r4, total_budget_r4, team_play_cnt_r4, total_team_player_r4, player_df_r4,
        # Round 5
        points_r5, price_r5, weight_r5, in_team_r5, available_r5, wk_weight_r5, bat_weight_r5, bowl_weight_r5,
        play_cnt_r5, total_player_r5, wk_cnt_r5, total_wk_r5, bat_cnt_r5, total_bat_r5, bowl_cnt_r5, total_bowl_r5,
        budget_r5, total_budget_r5, team_play_cnt_r5, total_team_player_r5, player_df_r5,
        # Round 6
        points_r6, price_r6, weight_r6, in_team_r6, available_r6, wk_weight_r6, bat_weight_r6, bowl_weight_r6,
        play_cnt_r6, total_player_r6, wk_cnt_r6, total_wk_r6, bat_cnt_r6, total_bat_r6, bowl_cnt_r6, total_bowl_r6,
        budget_r6, total_budget_r6, team_play_cnt_r6, total_team_player_r6, player_df_r6,
        # Round 7
        points_r7, price_r7, weight_r7, in_team_r7, available_r7, wk_weight_r7, bat_weight_r7, bowl_weight_r7,
        play_cnt_r7, total_player_r7, wk_cnt_r7, total_wk_r7, bat_cnt_r7, total_bat_r7, bowl_cnt_r7, total_bowl_r7,
        budget_r7, total_budget_r7, team_play_cnt_r7, total_team_player_r7, player_df_r7,
        # Round 8
        points_r8, price_r8, weight_r8, in_team_r8, available_r8, wk_weight_r8, bat_weight_r8, bowl_weight_r8,
        play_cnt_r8, total_player_r8, wk_cnt_r8, total_wk_r8, bat_cnt_r8, total_bat_r8, bowl_cnt_r8, total_bowl_r8,
        budget_r8, total_budget_r8, team_play_cnt_r8, total_team_player_r8, player_df_r8,
        # Round 9
        points_r9, price_r9, weight_r9, in_team_r9, available_r9, wk_weight_r9, bat_weight_r9, bowl_weight_r9,
        play_cnt_r9, total_player_r9, wk_cnt_r9, total_wk_r9, bat_cnt_r9, total_bat_r9, bowl_cnt_r9, total_bowl_r9,
        budget_r9, total_budget_r9, team_play_cnt_r9, total_team_player_r9, player_df_r9,
    )

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team.csv'), index=False)

----- Optimal Team Selection Summary -----
----- Round 1 -----
Total Expected Points (rnd 1): 1151.167252
Total Team Cost (rnd 1): 1764600.0
Captain (rnd 1): Ben Dwarshuis
Current Players Remaining (rnd 1): 0.0
----- Round 2 -----
Total Expected Points (rnd 2): 941.962479
Total Team Cost (rnd 2): 1673682.1241608458
Captain (rnd 2): Daniel Sams
Current Players Remaining (rnd 2): 9
----- Round 3 -----
Total Expected Points (rnd 3): 1035.395074
Total Team Cost (rnd 3): 1793695.832904919
Captain (rnd 3): Mitch Owen
Current Players Remaining (rnd 3): 9
----- Round 4 -----
Total Expected Points (rnd 4): 993.9825639999999
Total Team Cost (rnd 4): 1793633.5118553317
Captain (rnd 4): Cooper Connolly
Current Players Remaining (rnd 4): 8
----- Round 5 -----
Total Expected Points (rnd 5): 890.35311
Total Team Cost (rnd 5): 1802322.0238576205
Captain (rnd 5): Tom Curran
Current Players Remaining (rnd 5): 8
----- Round 6 -----
Total Expected Points (rnd 6): 920.8696100000001
Total Team Cost (rnd 6):

c:\Users\dilan\OneDrive\Documents\Data Science Projects\Repos\Big-Bash-Fantasy-AI-v2\python_script\pre-season\Optimisation_Functions.py:362: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel_player_df_r2['In_Team'] = sel_player_df_r2['Name'].astype(str).isin(sel_names_r1).astype(int)
c:\Users\dilan\OneDrive\Documents\Data Science Projects\Repos\Big-Bash-Fantasy-AI-v2\python_script\pre-season\Optimisation_Functions.py:374: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel_player_df_r3['In_Team'] = sel_player

# 2. New Distribution Approach

# a. Data Extraction 

In [30]:
if opt_strat == 'sim':
    # Simulated Player Points Data Constructions
    # Extract Player Expected Points and Standard Deviation dataframe
    model_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)
    sim_pts_df = model_pts_df.copy()

    # Generate Simulated Points based on Normal Distribution
    # Function to calculate z score bounds given confidence interval
    def z_score_bounds(confidence_level):
        """
        Returns the lower and upper z-scores for a given confidence level.
        For example, confidence_level = 0.90 for 90%.
        """
        alpha = 1 - confidence_level
        lower = norm.ppf(alpha / 2)
        upper = norm.ppf(1 - alpha / 2)
        return lower, upper

    # Set confidence interval and calculate z score bounds
    conf_int = 0.9
    lower_z_thresh, upper_z_thresh = z_score_bounds(conf_int)
    print(f"For {conf_int*100:.0f}% simulated points confidence interval the lower z score is {lower_z_thresh:.3f} and upper z score is {upper_z_thresh:.3f}")

    # Assign random z score for each df row within bounds and calculate simulated points
    sim_pts_df["z_score"] = np.random.uniform(lower_z_thresh, upper_z_thresh, size=len(sim_pts_df))
    sim_pts_df["sim_points"] = sim_pts_df["mean"] + (sim_pts_df["z_score"] * sim_pts_df["std_dev"])
    sim_pts_df["sim_points"] = sim_pts_df["sim_points"].clip(lower=0).round(0)  # Ensure no negative points

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")

Change opt_strat to sim for player fp distribution Optimisation Strategy


In [31]:
# New Optimisation Strategy (Player Fantasy Point Distribution)
if opt_strat == 'sim':
    # Data Extraction 
    # Current Round & Optimal Number of Rounds
    current_round = current_rnd
    opt_round = 2

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    # team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()
    # team_fix_df = team_fix_df[team_fix_df.Round <= current_round + opt_round - 1].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df, team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    # sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False).drop(["Unnamed: 0", "player"], axis = 1)
    sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, sim_pts_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], how = "left")
    player_df_raw["sim_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["sim_points"] + 10.51, player_df_raw["sim_points"] + 4.53)
    player_df_raw["weight"] = 1

    # Aggregate by player name (calculate total expected points for each player for x number of rounds)
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    sim_points=('sim_points',"sum"))

    # Create dataframe for next round expected points (For captain selection)
    player_df_next_rnd = player_df_raw[player_df_raw.Round == current_round].groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    next_rnd_sim_points=('sim_points',"sum"))
    player_df_next_rnd = player_df_next_rnd[["Name", "Team", "next_rnd_sim_points"]]

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")


Change opt_strat to sim for player fp distribution Optimisation Strategy


# 2. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [32]:
# Sim FP Whole Tournament Optimisation
if opt_strat == 'sim':
all_sim_sel_players = optimise_fn_sim_fp(conf_int, sim_num, player_df_raw)

IndentationError: expected an indented block after 'if' statement on line 2 (3043887311.py, line 3)

In [ ]:
# Player Distribution Optimisation Variable Setup
if opt_strat == 'sim':
    points = player_df["sim_points"]
    price = player_df["Price"]

    weight = player_df["weight"]
    available = player_df["Available"]
    in_team = player_df["In_Team"]
    wk_weight = player_df["Wk_f"]
    bat_weight = player_df["Bat_f"]
    bowl_weight = player_df["Bowl_f"]

    play_cnt, total_player = 12, range(len(price))
    # team_play_cnt, total_team_player = 0, range(len(price)) # IGNORE as pre tourny no players in team
    wk_cnt, total_wk = 1, range(len(price))
    bat_cnt, total_bat = 6, range(len(price))
    bowl_cnt, total_bowl = 5, range(len(price))
    budget, total_budget = 1783500, range(len(price))

    # MIP Optimsation Setup
    m = Model("knapsack")
    x = [m.add_var(var_type=BINARY) for i in total_player]
    print(x)

    m.objective = maximize(xsum(points[i]*x[i] for i in total_player))

    m += xsum(weight[i] * x[i] for i in total_player) == play_cnt
    m += xsum(wk_weight[i] * x[i] for i in total_wk) >= wk_cnt
    m += xsum(bat_weight[i] * x[i] for i in total_bat) >= bat_cnt
    m += xsum(bowl_weight[i] * x[i] for i in total_bowl) >= bowl_cnt
    m += xsum(price[i] * x[i] for i in total_budget) <= budget
    m += xsum(available[i] * x[i] for i in total_player) == play_cnt
    # m += xsum(in_team[i] * x[i] for i in total_team_player) >= team_play_cnt

    # Optimisation Process & Results
    m.optimize() 
    selected = [i for i in total_player if x[i].x >= 0.99]
    print("selected items: {}".format(selected))

    # Optimal Team Output
    sel_player_df = player_df.iloc[selected]
    sel_player_df = pd.merge(sel_player_df, player_df_next_rnd, left_on = ["Name","Team"], right_on = ["Name","Team"], how = "left")
    print("Total Simulated Points:", sum(sel_player_df["sim_points"]))
    print("Total Next Round Points:", sum(sel_player_df["next_rnd_sim_points"]))
    print("Total Team Cost:", sum(sel_player_df["Price"]))
    print("Number of Wk:", sum(sel_player_df["Wk_f"]))
    print("Number of Bat:", sum(sel_player_df["Bat_f"]))
    print("Number of Bowl:", sum(sel_player_df["Bowl_f"]))
    print("Available Players:", sum(sel_player_df["Available"]))
    print("Current Players Remaining:", sum(sel_player_df["In_Team"]))

    print(sel_player_df.sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 1].sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 0].sort_values(by = "Price", ascending = False))

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'EFP_optimal_pre_tourny_team.csv'))

else:
    print("Change opt_strat to sim for player distribution Optimisation Strategy")

Change opt_strat to sim for player distribution Optimisation Strategy
